# 01: Pipeline overview

**What this shows.** The probing pipeline in one diagram, then the smallest end-to-end sample of it: a single utterance pushed through MUSE, with per-layer CKA between clean and noisy activations read out from the demo parquet.

**What to look for.** A clean utterance $x_s$ and its degraded version $y$ both pass through the (frozen) SE network. We harvest activations $\mathbf{A}^{(\ell)}$ at every probed layer $\ell$, time-pool them into representations $\mathbf{H}^{(\ell)}$, and compare clean vs degraded with linear CKA. The output is a similarity score per layer per condition.

**Runtime.** <30 s on any device (reads the demo parquet). Optional inference cell at the bottom adds ~30 s on CUDA, ~2 min on MPS, ~5 min on CPU.

![Pipeline overview](figures/pipeline_flow.png)

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
while REPO.name != "SE-Probe" and REPO.parent != REPO:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from se_probe.device import device_info, get_device
from se_probe.plotting import MODEL_COLORS, MODEL_LABELS, apply_paper_rcparams

apply_paper_rcparams()

DEVICE = get_device()
print(device_info(DEVICE))

import pandas as pd

DATA_DIR = REPO / ("results_df" if (REPO / "results_df").exists() else "results_demo")
print(f"Using data from: {DATA_DIR}")

## One utterance, one SNR, one model

We pick a single clean utterance (`clean_idx`) and a single SNR for MUSE on the TBUS noise. The demo parquet ships precomputed CKA values; if `results_df/` is present the notebook silently uses the full table instead.

In [ ]:
muse_path = DATA_DIR / ("snr/cka_snr_muse.parquet" if (DATA_DIR / "snr").exists() else "cka_snr_muse_demo.parquet")
df = pd.read_parquet(muse_path)
df = df[df["noise_name"] == "TBUS"].copy()

first = df.iloc[0]
utt, snr = int(first["clean_idx"]), float(first["snr"])
one = df[(df["clean_idx"] == utt) & (df["snr"] == snr)].sort_values("layer").reset_index(drop=True)
print(f"utterance clean_idx={utt}, SNR={snr:g} dB, layers={len(one)}")
one[["layer", "CKA"]].head(8)

In [ ]:
first_layer_cka = float(one["CKA"].iloc[0])
last_layer_cka = float(one["CKA"].iloc[-1])
min_layer_cka = float(one["CKA"].min())
print(f"CKA at first probed layer: {first_layer_cka:.4f}")
print(f"CKA at last probed layer:  {last_layer_cka:.4f}")
print(f"CKA minimum across layers: {min_layer_cka:.4f}")

## Optional: live inference

Set `SE_PROBE_RUN_INFERENCE=1` (and have run `python scripts/setup.py` to fetch the upstream MUSE weights) to recompute one CKA value from raw audio. The cell loads the model, mixes a synthetic noisy clip, and runs a single forward pass through the activation extractor. Skipped by default.

In [ ]:
import os

if os.environ.get("SE_PROBE_RUN_INFERENCE") == "1":
    import numpy as np

    from se_probe.activation_extraction import load_muse_activation_extractor
    from se_probe.cka import cka

    rng = np.random.default_rng(0)
    sr = 16000
    clean = rng.standard_normal(sr * 2).astype("float32") * 0.05
    noise = rng.standard_normal(sr * 2).astype("float32") * 0.05
    target_snr_db = 0.0
    sig_p = float((clean ** 2).mean())
    noise_p = sig_p / (10 ** (target_snr_db / 10))
    noisy = clean + noise * np.sqrt(noise_p / max(float((noise ** 2).mean()), 1e-12))

    extractor = load_muse_activation_extractor(device=DEVICE)
    act_clean, _ = extractor(clean)
    act_noisy, _ = extractor(noisy)

    layer = next(iter(act_clean))
    val = cka(act_clean[layer], act_noisy[layer])
    print(f"Live CKA on synthetic clip ({layer}): {float(val):.4f}")
else:
    print("SE_PROBE_RUN_INFERENCE not set — skipping live inference. Set it to '1' to enable.")